Import libraries

In [38]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [39]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/train_women.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/val_women.csv")

CPU times: user 24.4 ms, sys: 8.02 ms, total: 32.5 ms
Wall time: 32 ms
CPU times: user 11.5 ms, sys: 4.01 ms, total: 15.5 ms
Wall time: 15.4 ms
CPU times: user 4.14 ms, sys: 1 ms, total: 5.14 ms
Wall time: 5.1 ms


In [40]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [41]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [42]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.78953321  1.6460815   1.41745802 -1.11692492  0.91314538  1.07643872
   0.9950542  -0.36894675 -0.88924565  1.13133938]]


In [43]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [44]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25615, 10]); y=torch.Size([25615, 1]); t=torch.Size([25615, 1])
Shape of val: x=torch.Size([4270, 10]); y=torch.Size([4270, 1]); t=torch.Size([4270, 1])
Shape of test: x=torch.Size([12808, 10]); y=torch.Size([12808, 1]); t=torch.Size([12808, 1])


In [45]:
epochs = 150
lr = 5.5e-5
wd = 1e-5
alpha = 1
beta= 1
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
shared_dropout = 0
outcome_dropout = 0.0
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")
print (f" activation = {activation}")

 epochs = 150
 learning rate = 5.5e-05
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30
 activation = <class 'torch.nn.modules.activation.ReLU'>


In [46]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_alpha = trial.suggest_categorical("alpha", [0.1, 0.5, 1.0])
    grid_beta = trial.suggest_categorical("beta", [0.1, 0.5, 1.0])

    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        dragonnet = Dragonnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            beta=grid_beta,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_droupout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence Dragonnet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            dragonnet.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p, _, _ = dragonnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "alpha": t.params["alpha"],
        "beta": t.params["beta"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "alpha": float(best_params["alpha"]),
    "beta": float(best_params["beta"]),
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 42


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
Best trial: 49. Best value: 0.87023: 100%|██████████| 50/50 [42:09<00:00, 50.58s/it] 


In [50]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 49
Best mean_Val_AUQC: 0.870230
Best params: {'lr': 0.00024260106038418808, 'weight_decay': 1.539684304296834e-05, 'alpha': 0.5, 'beta': 1.0}

Best config table:


,lr,weight_decay,alpha,beta,mean_Val_AUQC,std_Val_AUQC
0,0.000243,0.000015,0.5,1.0,0.87023,0.070152



Top 10 trials:


,trial,lr,weight_decay,alpha,beta,mean_Val_AUQC,std_Val_AUQC
0,49,0.000243,0.000015,0.5,1.0,0.870230,0.070152
1,22,0.000276,0.000015,0.1,1.0,0.870023,0.069164
2,43,0.000277,0.000019,0.1,1.0,0.869389,0.069659
3,19,0.000246,0.000015,0.1,1.0,0.868733,0.069016
4,12,0.000256,0.000059,0.1,1.0,0.867887,0.073311
5,41,0.000309,0.000012,0.1,1.0,0.866218,0.074604
6,32,0.000260,0.000014,0.1,1.0,0.866048,0.063918
7,33,0.000305,0.000012,0.1,1.0,0.865899,0.073997
8,31,0.000257,0.000014,0.1,1.0,0.865247,0.064728
9,24,0.000257,0.000016,0.1,1.0,0.865159,0.064300



Per-seed test results:


,Seed,AUUC,AUQC,Lift,KRCC,ATE_Err
0,412312,0.848680,0.854274,0.006596,0.094056,0.015931
1,42,0.561338,0.557980,0.003907,0.038982,0.017221
2,1874,0.642258,0.640964,0.005246,0.113203,0.574409
3,902745,0.744389,0.740547,0.005710,0.118488,0.131346
4,1,0.900796,0.906985,0.006835,0.197447,0.017788



Test metrics mean ± std:


,AUUC,AUQC,Lift,KRCC,ATE_Err
mean,0.739492,0.740150,0.005659,0.112435,0.151339
std,0.140675,0.144812,0.001173,0.057003,0.241633


In [51]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_alpha = float(best_cfg['alpha'])
best_beta = float(best_cfg['beta'])

print("Đang đánh giá trên test với best config từ validation:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}, alpha={best_alpha:.1f}, beta={best_beta:.1f}")
print(f"Số seed: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    dragonnet = Dragonnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        alpha=best_alpha,
        beta=best_beta,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=shared_hidden,
        outcome_hidden=outcome_hidden,
        outcome_droupout=outcome_dropout,
        shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
        activation=activation
    )

    dragonnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred, _, _ = dragonnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'CHI TIẾT TỪNG SEED (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'KẾT QUẢ TRUNG BÌNH TEST (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Đang đánh giá trên test với best config từ validation:
  lr=2.4e-04, weight_decay=1.5e-05, alpha=0.5, beta=1.0
Số seed: 5
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Base Loss: 1.6785 | Tarreg Loss: 1.852010 | Total Loss: 3.5305 | Val Loss: 1.6733 | Raw Qini: 0.8176 | EMA Trend: 0.8176 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.6868 | Tarreg Loss: 1.403322 | Total Loss: 3.0901 | Val Loss: 1.6429 | Raw Qini: 0.6735 | EMA Trend: 0.7960 | (patience: 1/20)
Epoch 3/150 | Base Loss: 1.6107 | Tarreg Loss: 1.788515 | Total Loss: 3.3992 | Val Loss: 1.6071 | Raw Qini: 0.6526 | EMA Trend: 0.7745 | (patience: 2/20)
Epoch 4/150 | Base Loss: 1.5549 | Tarreg Loss: 1.835159 | Total Loss: 3.3900 | Val Loss: 1.5586 | Raw Qini: 0.5974 | EMA

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 1.7721 | Tarreg Loss: 1.238386 | Total Loss: 3.0105 | Val Loss: 1.7800 | Raw Qini: 0.5876 | EMA Trend: 0.5876 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.7317 | Tarreg Loss: 1.091770 | Total Loss: 2.8235 | Val Loss: 1.7366 | Raw Qini: 0.8350 | EMA Trend: 0.6247 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 1.6934 | Tarreg Loss: 0.990272 | Total Loss: 2.6837 | Val Loss: 1.6791 | Raw Qini: 0.9707 | EMA Trend: 0.6766 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 1.6282 | Tarreg Loss: 1.252940 | Total Loss: 2.8811 | Val Loss: 1.6056 | Raw Qini: 0.9367 | EMA Trend: 0.7156 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Base Loss: 1.4680 | Tarreg Loss: 1.097519 | Total Loss: 2.5655 | Val Loss: 1.4997 | Raw Qini: 0.9274 | EMA Trend: 0.7474 | ✓ above trend but not peak (patience: 2/20)
Epoch 6/150 | Base Loss: 1.4053 | Tarreg Loss: 0.796928 | Total Loss: 2.2023 | Val Loss: 1.3416 | Raw Qini: 0.9246 | EMA Trend: 0.7740 | ✓ above tren

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 1.7633 | Tarreg Loss: 9.318486 | Total Loss: 11.0818 | Val Loss: 1.7713 | Raw Qini: 0.6071 | EMA Trend: 0.6071 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.7656 | Tarreg Loss: 9.596092 | Total Loss: 11.3617 | Val Loss: 1.7694 | Raw Qini: 0.6810 | EMA Trend: 0.6182 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 1.7568 | Tarreg Loss: 9.803747 | Total Loss: 11.5606 | Val Loss: 1.7703 | Raw Qini: 0.7130 | EMA Trend: 0.6324 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 1.7758 | Tarreg Loss: 9.213513 | Total Loss: 10.9893 | Val Loss: 1.7730 | Raw Qini: 0.7554 | EMA Trend: 0.6509 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 1.7712 | Tarreg Loss: 9.441312 | Total Loss: 11.2125 | Val Loss: 1.7763 | Raw Qini: 0.7839 | EMA Trend: 0.6708 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 1.7974 | Tarreg Loss: 8.396999 | Total Loss: 10.1944 | Val Loss: 1.7774 | Raw Qini: 0.7898 | EMA Trend: 0.6887 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Ba

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 1.7784 | Tarreg Loss: 3.986679 | Total Loss: 5.7651 | Val Loss: 1.7819 | Raw Qini: 0.1581 | EMA Trend: 0.1581 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.7520 | Tarreg Loss: 3.518201 | Total Loss: 5.2702 | Val Loss: 1.7460 | Raw Qini: 0.2141 | EMA Trend: 0.1665 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 1.7179 | Tarreg Loss: 3.359754 | Total Loss: 5.0777 | Val Loss: 1.7038 | Raw Qini: 0.3285 | EMA Trend: 0.1908 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 1.6775 | Tarreg Loss: 3.063706 | Total Loss: 4.7413 | Val Loss: 1.6543 | Raw Qini: 0.5866 | EMA Trend: 0.2501 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 1.6020 | Tarreg Loss: 3.443401 | Total Loss: 5.0454 | Val Loss: 1.5897 | Raw Qini: 0.7837 | EMA Trend: 0.3302 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 1.5048 | Tarreg Loss: 3.046003 | Total Loss: 4.5509 | Val Loss: 1.4888 | Raw Qini: 0.8457 | EMA Trend: 0.4075 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Base Los

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 1.6412 | Tarreg Loss: 0.774779 | Total Loss: 2.4160 | Val Loss: 1.6333 | Raw Qini: 0.6741 | EMA Trend: 0.6741 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 1.6179 | Tarreg Loss: 0.649550 | Total Loss: 2.2674 | Val Loss: 1.5964 | Raw Qini: 0.8234 | EMA Trend: 0.6965 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 1.5611 | Tarreg Loss: 0.680170 | Total Loss: 2.2412 | Val Loss: 1.5450 | Raw Qini: 0.7708 | EMA Trend: 0.7076 | ✓ above trend but not peak (patience: 1/20)
Epoch 4/150 | Base Loss: 1.4647 | Tarreg Loss: 0.968955 | Total Loss: 2.4337 | Val Loss: 1.4711 | Raw Qini: 0.6648 | EMA Trend: 0.7012 | (patience: 2/20)
Epoch 5/150 | Base Loss: 1.4396 | Tarreg Loss: 0.549255 | Total Loss: 1.9889 | Val Loss: 1.3671 | Raw Qini: 0.6001 | EMA Trend: 0.6860 | (patience: 3/20)
Epoch 6/150 | Base Loss: 1.2348 | Tarreg Loss: 0.647608 | Total Loss: 1.8824 | Val Loss: 1.2230 | Raw Qini: 0.6374 | EMA Trend: 0.6787 | (patience: 4/20)
Epoch 7/150 | Base Loss: 1.0854

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/Dragonnet/dragonnet.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
